# Clasificación Binaria: ¿Este empleado va a dejar la empresa?

Regresión logística aplicada a datos de RRHH para predecir fuga de talento.
Primeros pasos en clasificación: confusión matrix, ROC, AUC, threshold tuning.

**Dataset:** [IBM HR Analytics Employee Attrition](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)  
**Autor:** @aroaxinping  
**Fecha:** Abril 2026

---
## 0. Configuración del entorno

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve,
    accuracy_score, f1_score,
)

# Estilo
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'text.color':       '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
    'axes.titlecolor':  '#ffffff',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

ACCENT  = '#e85d04'
ACCENT2 = '#6a9ad4'
ACCENT3 = '#2dc653'
WARN    = '#f4d03f'

print('Entorno listo.')

---
## 1. Datos

| Dataset | Fuente | Registros |
|---|---|---|
| IBM HR Analytics | [Kaggle](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset) | 1470 empleados |

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

DATA_PATH = Path('../data/processed/attrition_clean.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    es_sintetico = False
    print(f'[OK] Datos cargados: {len(df)} empleados')
else:
    print('[INFO] Datos no encontrados. Generando dataset sintético...')
    print('[TIP]  Ejecuta: python src/fetch_attrition.py')
    from fetch_attrition import generate_synthetic_attrition
    df = generate_synthetic_attrition()
    es_sintetico = True

if es_sintetico:
    print('\n⚠️  AVISO: Datos sintéticos.')

# Target binario
df['Attrition_bin'] = (df['Attrition'] == 'Yes').astype(int)
TARGET = 'Attrition_bin'

print(f'\nAttrition rate: {df[TARGET].mean()*100:.1f}%')
print(f'Shape: {df.shape}')
df.head()

---
## 2. Exploración

> **Pregunta:** ¿Qué factores están más asociados con la fuga de empleados?

In [ ]:
# 2.1 Balance de clases
print('--- Balance de clases ---')
print(df['Attrition'].value_counts())
print(f'\nRatio No:Yes = {(df[TARGET]==0).sum() / max((df[TARGET]==1).sum(), 1):.1f}:1')
print('→ Dataset desbalanceado. Hay que tenerlo en cuenta en las métricas.')

In [ ]:
# 2.2 Features numéricas: distribución por attrition
num_features = [
    'Age', 'MonthlyIncome', 'DistanceFromHome', 'YearsAtCompany',
    'TotalWorkingYears', 'NumCompaniesWorked', 'JobSatisfaction',
    'WorkLifeBalance', 'TrainingTimesLastYear',
]
# Filtrar a las que existen en el df
num_features = [f for f in num_features if f in df.columns]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, feat in zip(axes.flat, num_features):
    for val, color, label in [(0, ACCENT2, 'Se queda'), (1, ACCENT, 'Se va')]:
        data = df[df[TARGET] == val][feat].dropna()
        ax.hist(data, bins=20, alpha=0.6, color=color, label=label, density=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=7)
for ax in axes.flat[len(num_features):]:
    ax.set_visible(False)
plt.suptitle('Distribución por Attrition', y=1.01, color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 OverTime vs Attrition
if 'OverTime' in df.columns:
    ct = pd.crosstab(df['OverTime'], df['Attrition'], normalize='index') * 100
    print('--- Attrition rate por OverTime ---')
    print(ct.round(1))
    print(f'\nHacer horas extra multiplica por {ct.loc["Yes", "Yes"] / max(ct.loc["No", "Yes"], 0.1):.1f}x la probabilidad de irse.')

---
## 3. Preparación de datos

> **Pregunta:** ¿Cómo codificar variables categóricas y escalar las numéricas para logistic regression?

In [ ]:
# 3.1 Seleccionar features
cat_features = ['OverTime', 'MaritalStatus', 'Department', 'Gender']
cat_features = [f for f in cat_features if f in df.columns]

all_features = num_features + cat_features
print(f'Features numéricas: {num_features}')
print(f'Features categóricas: {cat_features}')

# One-hot encoding
df_model = pd.get_dummies(df[all_features + [TARGET]], columns=cat_features, drop_first=True)
feature_cols = [c for c in df_model.columns if c != TARGET]

print(f'\nFeatures finales ({len(feature_cols)}): {feature_cols}')

In [ ]:
# 3.2 Train/test split + scaling
X = df_model[feature_cols]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_test_sc = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)

print(f'Train: {len(X_train)} ({y_train.mean()*100:.1f}% attrition)')
print(f'Test:  {len(X_test)} ({y_test.mean()*100:.1f}% attrition)')

---
## 4. Regresión Logística

> **Pregunta:** ¿Qué tan bien predice un modelo logístico si un empleado se va a ir?

In [ ]:
# 4.1 Fit
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

y_pred = lr.predict(X_test_sc)
y_prob = lr.predict_proba(X_test_sc)[:, 1]

print('--- Resultados (threshold = 0.5) ---')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(f'AUC:      {roc_auc_score(y_test, y_prob):.3f}')
print(f'F1:       {f1_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=['Se queda', 'Se va']))

In [ ]:
# 4.2 Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Oranges',
    xticklabels=['Se queda', 'Se va'],
    yticklabels=['Se queda', 'Se va'],
    ax=ax, linewidths=1, linecolor='#333',
)
ax.set(xlabel='Predicción', ylabel='Real', title='Confusion Matrix')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Positives (detectados):  {tp}')
print(f'False Negatives (se escapan): {fn}')
print(f'False Positives (falsas alarmas): {fp}')

---
## 5. Curva ROC y AUC

> **Pregunta:** ¿Qué capacidad discriminativa tiene el modelo más allá del threshold de 0.5?

In [ ]:
# 5.1 ROC curve
fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
ax = axes[0]
ax.plot(fpr, tpr, color=ACCENT, lw=2, label=f'Logistic Regression (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], '--', color='#555', lw=1, label='Random (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.15, color=ACCENT)
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='Curva ROC')
ax.legend(loc='lower right')

# Precision-Recall
precision, recall, thresholds_pr = precision_recall_curve(y_test, y_prob)
ax = axes[1]
ax.plot(recall, precision, color=ACCENT2, lw=2)
ax.fill_between(recall, precision, alpha=0.15, color=ACCENT2)
baseline = y_test.mean()
ax.axhline(baseline, ls='--', color='#555', lw=1, label=f'Baseline = {baseline:.2f}')
ax.set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
ax.legend()

plt.tight_layout()
plt.show()

---
## 6. Threshold Tuning

> **Pregunta:** ¿Podemos mejorar el recall bajando el threshold? ¿A qué coste?

In [ ]:
# 6.1 Métricas en función del threshold
thresholds = np.arange(0.1, 0.9, 0.05)
metrics_by_th = []
for th in thresholds:
    y_th = (y_prob >= th).astype(int)
    cm_th = confusion_matrix(y_test, y_th)
    tn, fp, fn, tp = cm_th.ravel()
    metrics_by_th.append({
        'threshold': th,
        'accuracy': accuracy_score(y_test, y_th),
        'precision': tp / max(tp + fp, 1),
        'recall': tp / max(tp + fn, 1),
        'f1': f1_score(y_test, y_th),
        'false_alarms': fp,
    })
th_df = pd.DataFrame(metrics_by_th)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(th_df['threshold'], th_df['precision'], color=ACCENT, lw=2, label='Precision')
ax.plot(th_df['threshold'], th_df['recall'], color=ACCENT2, lw=2, label='Recall')
ax.plot(th_df['threshold'], th_df['f1'], color=ACCENT3, lw=2, label='F1')
ax.axvline(0.5, ls='--', color='#555', lw=1, label='Default (0.5)')

# Marcar el mejor F1
best_th = th_df.loc[th_df['f1'].idxmax(), 'threshold']
ax.axvline(best_th, ls='--', color=WARN, lw=1.5, label=f'Mejor F1 (th={best_th:.2f})')

ax.set(xlabel='Threshold', ylabel='Score', title='Precision / Recall / F1 vs Threshold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nMejor threshold por F1: {best_th:.2f}')
print(th_df[th_df['threshold'].round(2) == round(best_th, 2)].to_markdown(index=False))

---
## 7. Feature Importance (Odds Ratios)

> **Pregunta:** ¿Qué factores aumentan más la probabilidad de irse?

In [ ]:
# 7.1 Odds ratios
odds = pd.DataFrame({
    'feature': feature_cols,
    'coef': lr.coef_[0],
    'odds_ratio': np.exp(lr.coef_[0]),
}).sort_values('odds_ratio', ascending=True)

fig, ax = plt.subplots(figsize=(8, 7))
colors = [ACCENT if o > 1 else ACCENT2 for o in odds['odds_ratio']]
ax.barh(odds['feature'], odds['odds_ratio'], color=colors, edgecolor='#333')
ax.axvline(1.0, color=WARN, ls='--', lw=1.5, label='Sin efecto (OR=1)')
ax.set(xlabel='Odds Ratio', title='Odds Ratios — Regresión Logística')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=ACCENT, label='↑ Más probabilidad de irse'),
    Patch(color=ACCENT2, label='↓ Menos probabilidad de irse'),
], loc='lower right')
plt.tight_layout()
plt.show()

print('\n--- Top 5 factores de riesgo ---')
top_risk = odds.nlargest(5, 'odds_ratio')
for _, row in top_risk.iterrows():
    print(f"  {row['feature']:30s} OR = {row['odds_ratio']:.2f} (x{row['odds_ratio']:.1f} probabilidad)")

---
## 8. Cross-Validation

> **Pregunta:** ¿Los resultados son estables o dependen del split?

In [ ]:
# 8.1 5-Fold CV
X_all_sc = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

cv_accuracy = cross_val_score(lr, X_all_sc, y, cv=5, scoring='accuracy')
cv_auc = cross_val_score(lr, X_all_sc, y, cv=5, scoring='roc_auc')
cv_f1 = cross_val_score(lr, X_all_sc, y, cv=5, scoring='f1')

print('--- 5-Fold Cross-Validation ---')
print(f'  Accuracy: {cv_accuracy.mean():.3f} ± {cv_accuracy.std():.3f}')
print(f'  AUC:      {cv_auc.mean():.3f} ± {cv_auc.std():.3f}')
print(f'  F1:       {cv_f1.mean():.3f} ± {cv_f1.std():.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
x_pos = np.arange(3)
means = [cv_accuracy.mean(), cv_auc.mean(), cv_f1.mean()]
stds = [cv_accuracy.std(), cv_auc.std(), cv_f1.std()]
bars = ax.bar(x_pos, means, yerr=stds, color=[ACCENT, ACCENT2, ACCENT3],
              edgecolor='#333', capsize=5)
ax.set_xticks(x_pos)
ax.set_xticklabels(['Accuracy', 'AUC', 'F1'])
ax.set(ylabel='Score', title='Cross-Validation (5-Fold)', ylim=(0, 1))
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{m:.3f}', ha='center', color='#e0e0e0', fontsize=11)
plt.tight_layout()
plt.show()

---
## 9. Síntesis y conclusiones

In [ ]:
# 9.1 Tabla resumen
best_row = th_df.loc[th_df['f1'].idxmax()]
resumen = pd.DataFrame([
    {
        'Métrica': 'Accuracy',
        'Threshold 0.5': f"{accuracy_score(y_test, y_pred):.3f}",
        f'Threshold {best_th:.2f}': f"{best_row['accuracy']:.3f}",
        'CV (5-fold)': f"{cv_accuracy.mean():.3f} ± {cv_accuracy.std():.3f}",
    },
    {
        'Métrica': 'AUC',
        'Threshold 0.5': f"{auc:.3f}",
        f'Threshold {best_th:.2f}': f"{auc:.3f}",
        'CV (5-fold)': f"{cv_auc.mean():.3f} ± {cv_auc.std():.3f}",
    },
    {
        'Métrica': 'F1',
        'Threshold 0.5': f"{f1_score(y_test, y_pred):.3f}",
        f'Threshold {best_th:.2f}': f"{best_row['f1']:.3f}",
        'CV (5-fold)': f"{cv_f1.mean():.3f} ± {cv_f1.std():.3f}",
    },
])
print(resumen.to_markdown(index=False))

print('\n--- Conclusión ---')
print(f'AUC = {auc:.3f} — el modelo discrimina razonablemente entre los que se van y los que no.')
print(f'El accuracy es alto pero engañoso: con un 84% de "No", predecir siempre "No" ya da ~84%.')
print(f'El F1 es la métrica que importa en datasets desbalanceados.')
print(f'\nFactores de riesgo clave: OverTime, distancia, salario bajo, poca satisfacción.')
print(f'Próximo paso: ¿un modelo no lineal (árboles) captará mejor estos patrones?')